<a href="https://colab.research.google.com/github/tirthas970-cmyk/Galaxy-Classifier-Using-ML/blob/main/Galaxy_Classifier_Copy_for_Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NOTE:
# Quick Background: This is a galaxy classifer. Essentially, I'm using 24,000 images from the AstroNN dataset to train my model to classify galaxies based on their shapes using computer visions
# This not a completed project, and is in actively in development
# Here you will see my developmental process from my intial model to my current model
# To run, please press the run all button --> Note that this will run all models, but the main model (i.e., my current one) is in the last cell

In [ ]:
!pip install astroNN

In [ ]:
from astroNN.datasets import load_galaxy10sdss
import numpy as np
import tensorflow as tf
from keras import layers, models
from sklearn.model_selection import train_test_split

images, labels = load_galaxy10sdss()

Galaxy10.h5:  99%|█████████▊| 208M/210M [00:16<00:00, 14.8MB/s]

Downloaded Galaxy10 successfully to /root/.astroNN/datasets/Galaxy10.h5


Galaxy10.h5: 210MB [00:17, 12.1MB/s]                           


In [ ]:
#Normalize images into [0, 1]
images = images.astype(np.float32) / 255.0

#Number 5 does skew with data due to low amount of images
mask = (labels !=5)

images = images[mask]
labels = labels[mask]


# Split into 70/10/20
train_idx, test_idx = train_test_split(range(labels.shape[0]), test_size=0.2, random_state=42, stratify=labels)
train_idx, val_idx = train_test_split(train_idx, test_size=0.125, random_state=42, stratify=labels[train_idx])

# Create the actual datasets using the indices
train_images, train_labels = images[train_idx], labels[train_idx]
val_images, val_labels = images[val_idx], labels[val_idx]
test_images, test_labels = images[test_idx], labels[test_idx]

num_classes = 10

#Turns numpy array into tensorflow dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(1000).batch(16)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(16)

#Optimization
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Remaining classes: {np.unique(labels)}")
print(f"New dataset size: {len(images)}")

Remaining classes: [0 1 2 3 4 6 7 8 9]
New dataset size: 21768


In [ ]:
#Build CNN --> FIRST CNN

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"
  layers.RandomRotation(factor=0.5),            # Full 360-degree rotation (1.0 = 2π)
  layers.RandomZoom(0.2),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                   # Account for different telescope sensors


  layers.Conv2D(32, (3, 3), activation='relu'),

  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), activation='relu'),

  layers.MaxPooling2D((2,2)),

  layers.Flatten(),
  #To prevent overfitting
  layers.Dropout(0.5),

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

In [ ]:
###Refinement to Data Preperation

from sklearn.utils import class_weight

#Just changed batch sized from 16 to 32
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(1000).batch(64)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(64)

#Optimization
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

#Adding Class Weights to prevent overfitting on majority of data
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
sqrt_weights = np.sqrt(weights)

class_weight_dict = dict(zip(np.unique(train_labels), sqrt_weights))

print(class_weight_dict)




In [ ]:
###Refinement to Model

#Batchnormalization

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"
  layers.RandomRotation(factor=0.5),            # variations in rotations
  layers.RandomZoom(0.2),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors
  layers.RandomTranslation(0.1, 0.1),           #Create more variations


  layers.Conv2D(64, (3, 3), use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  #Prevents Overfitting
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.2),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

model.fit(train_ds, validation_data=val_ds, epochs=50, class_weight=class_weight_dict)

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

In [ ]:
### More Refinement to Model


model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors
 # layers.RandomTranslation(0.05, 0.05),           #Create more variations


  layers.Conv2D(64, (3, 3), padding='same', use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Flatten(), #Changed it back to flatten so that spacial dimensions info isn't lost
  layers.Dense(256, activation='relu'), #to catch more patterns
  layers.Dropout(0.5),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

In [ ]:
#See socres and how model is doing

import numpy as np
from sklearn.metrics import classification_report

all_labels = []
all_preds = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    all_labels.extend(labels.numpy())
    # Take the index of the highest probability
    all_preds.extend(np.argmax(preds, axis=1))

print(classification_report(all_labels, all_preds))

In [ ]:
###fine tunning
#Stratify data


model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors

  layers.Conv2D(64, (3, 3), padding='same', use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('swish'), #swish for more optimization
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),


  layers.GlobalAveragePooling2D(), # Testing out GlobalAveragePooling Again
  layers.Dense(256, activation='relu'), #to catch more patterns
  layers.Dropout(0.5),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])     #Wanted to test without class_weights

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

In [ ]:
# To Check scores of previous model

import numpy as np
from sklearn.metrics import classification_report

all_labels = []
all_preds = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    all_labels.extend(labels.numpy())
    all_preds.extend(np.argmax(preds, axis=1))

print(classification_report(all_labels, all_preds))

In [ ]:
from tensorflow.keras import regularizers
#Even More finetuning

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                    # Account for different telescope sensors

  layers.Conv2D(64, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),  #using kernal regularizer to prevent the model from just memorizing info
  layers.BatchNormalization(),
  layers.Activation('swish'), #swish for more optimization
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),


  layers.GlobalAveragePooling2D(),
  layers.Dense(256, activation='swish'), #to catch more patterns
  layers.Dropout(0.5),

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")